In [2]:
import os
import pandas as pd

current = os.getcwd()

while not os.path.exists(os.path.join(current, "data")):
    current = os.path.dirname(current)


data_path = os.path.join(current, "data", "processed", "train_processed.csv")
df = pd.read_csv(data_path)

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

X = df.drop("TARGET", axis=1)
y = df["TARGET"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

cat_cols = X_train.select_dtypes(include='object').columns

label_encoders = {}

for col in cat_cols:
    le = LabelEncoder()

    X_train[col] = le.fit_transform(X_train[col].astype(str))

    # Handle unseen categories in test
    X_test[col] = X_test[col].astype(str).map(
        lambda x: le.transform([x])[0] if x in le.classes_ else -1
    )

    label_encoders[col] = le

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    class_weight='balanced',
    n_jobs=-1,
    random_state=42
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nROC-AUC:", roc_auc_score(y_test, y_prob))


C:\Users\abeku\AppData\Local\Temp\ipykernel_30952\3371180721.py:19: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train.select_dtypes(include='object').columns


Confusion Matrix:
 [[43529 13009]
 [ 1952  3013]]

Classification Report:
               precision    recall  f1-score   support

           0       0.96      0.77      0.85     56538
           1       0.19      0.61      0.29      4965

    accuracy                           0.76     61503
   macro avg       0.57      0.69      0.57     61503
weighted avg       0.89      0.76      0.81     61503


ROC-AUC: 0.7577559026240387


### Hyperparameter tuning

In [ ]:
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.model_selection import RandomizedSearchCV
#
# # Model
# rf = RandomForestClassifier(random_state=42, n_jobs=1)
#
# # Parameter grid
# param_dist = {
#     "n_estimators": [100, 200, 300, 500],
#     "max_depth": [5, 8, 10, 15, 20],
#     "min_samples_split": [2, 5, 10, 20],
#     "min_samples_leaf": [5, 10, 20, 50],
#     "max_features": ["sqrt", "log2", 0.5, 0.8],
#     "class_weight": ["balanced"]
# }
#
# # Random search
# random_search = RandomizedSearchCV(
#     estimator=rf,
#     param_distributions=param_dist,
#     n_iter=20,
#     scoring="roc_auc",
#     cv=3,
#     verbose=2,
#     random_state=42,
#     n_jobs=1
# )
#
# random_search.fit(X_train, y_train)


Fitting 3 folds for each of 20 candidates, totalling 60 fits
[CV] END class_weight=balanced, max_depth=20, max_features=0.8, min_samples_leaf=10, min_samples_split=10, n_estimators=100; total time=22.4min
[CV] END class_weight=balanced, max_depth=20, max_features=0.8, min_samples_leaf=10, min_samples_split=10, n_estimators=100; total time=21.7min
